# Assignment: Barcelona

## Prep

### Imports, shared definitions, datasets

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily
import pointpats
import numpy as np

In [ ]:
listings_url = "https://data.insideairbnb.com/spain/catalonia/barcelona/2025-09-14/data/listings.csv.gz"
listings_df = pd.read_csv(listings_url, compression='gzip')
listings_df

## Part 2

In [ ]:
listings_geometry = gpd.points_from_xy(listings_df['longitude'], listings_df['latitude'], crs="EPSG:4326")
listings_geometry

In [ ]:
listings_gdf = gpd.GeoDataFrame(listings_df, geometry=listings_geometry)
listings_gdf.head()

In [ ]:
listings_gdf.explore(tiles="CartoDB Positron")

In [ ]:
# density

In [ ]:
f, ax = plt.subplots()
listings_gdf.plot(ax=ax, markersize=0.05)

In [ ]:
# Q: Are the Airbnb listings distributed equally across the city?
# No

f, ax = plt.subplots()
# listings_gdf.plot(ax=ax, markersize=0.05)
pointpats.plot_density(
    listings_gdf,
    bandwidth=500,
    levels=25,
    # alpha=0.55,
    # cmap="magma_r",
    # linewidths=1,
    ax=ax,
)
contextily.add_basemap(
    ax=ax,
    crs=listings_gdf.crs,
    source="CartoDB Positron No Labels",
)

In [ ]:
# Does it depend on the type of listing or its price?

In [ ]:
listings_gdf.columns


In [ ]:
listings_gdf["property_type"].head(20)

In [ ]:
listings_gdf.explore("property_type")

In [ ]:
listings_gdf["price"].head(10)

In [ ]:
listings_gdf["price_dollars"] = (
    listings_gdf["price"]
      .str.replace(r"[\$,]", "", regex=True)
      .astype(float)
)

In [ ]:
listings_gdf["price_dollars"].head(10)

In [ ]:
listings_gdf.explore("price_dollars")

# Thoughts / Plan of attack

Ok, answering "Does it depend on the type of listing or its price?" is not straightforward (at least for me) because "type of listing" is categorical whilst price is numerical.

Based on reading through bits of course again, summary of planned approach:

## price

1. Find all points which have a price defined (ignore NaN)
2. Get the convex cull of the points
3. Apply a hexbin over this area, at some approx radius
4. Map each price to its hexbin
5. Compute median price of each hexbin (using median here as there are some very large prices)
6. For hexbin, apply spatial autocorrelation, Moran's I?

## type of listing

1. Find all points which have a listing type
2. For each point, find it's nearest neighbour (using KNN); perhaps choose radius as same as approx radius of hexbin for price?
3. Compute a binary outcome by checking if each point has the same category as its neighbhour
4. Using Join Counts statistic from here

In [ ]:
def price_only(gdf):
    price_gdf = gdf.copy(deep=True)
    price_gdf["price"] = (
        gdf["price"]
          .str.replace(r"[\$,]", "", regex=True)
          .astype(float)
    )
    price_gdf = price_gdf[["price", gdf.geometry.name]]
    price_gdf = price_gdf.dropna()
    return price_gdf
    
listings_price_gdf = price_only(listings_gdf)
listings_price_gdf.head()

In [ ]:
listings_price_gdf.explore("price")

In [ ]:
listings_price_hull = listings_price_gdf.union_all()

In [ ]:
listings_price_hull

In [ ]:
import h3

def point_to_h3_fn(h3_res):
    """returns a new function which will convert a POINT geometry into an H3 id"""
    def f(row):
        return h3.latlng_to_cell(row.geometry.x, row.geometry.y, h3_res)
    return f

def add_grid_cells(gdf, h3_res):
    """adds a new `h3_id` column to a GDF which is assumed to have a POINT geometry"""
    gdf["h3_id"] = gdf.apply(point_to_h3_fn(h3_res), axis=1)
    
    

In [ ]:
add_grid_cells(listings_price_gdf, h3_res=8)
listings_price_gdf.head()

In [ ]:
from shapely.geometry import Polygon

def h3_to_polygon(h3_id):
    """takes an H3 and returns a boundary as a Shapely Polygon"""
    boundary = h3.cell_to_boundary(h3_id)
    lng_lat = [(lng, lat) for lat, lng in boundary]
    return Polygon(lng_lat)

def visualise_grid_cells(gdf):
    """takes a GDF with an `h3_id` column, which may contain duplicates, 
    and returns a new GDF with all the unique H3 cells as Polygons"""
    unique_ids = gdf['h3_id'].unique()
    polygon_gdf = gpd.GeoDataFrame(
        {'h3_id': unique_ids},
        geometry=[h3_to_polygon(h) for h in unique_ids],
        crs='EPSG:4326'
    )
    return polygon_gdf
    

In [ ]:
visualise_grid_cells(listings_price_gdf).explore()